# 01.7 · 优雅降级（Graceful Degradation）

> 部分 Agent 故障时仍能返回有用结果。


In [ ]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

---
## Part 7 · 优雅降级（Graceful Degradation）

**场景**：Reranker 服务不可用时，系统不应整体崩溃，而应**降级**继续提供服务。

### 四条路径总览

```mermaid
flowchart LR
    subgraph Normal["✅ 正常路径"]
        direction LR
        N1["retrieve"] --> N2["rerank"] --> N3["generate"]
    end

    subgraph D1["⚠️ 降级路径 1 — Reranker 故障"]
        direction LR
        R1["retrieve"] --> R2["skip rerank<br/>hits[:3]"] --> R3["generate"]
    end

    subgraph D2["⚠️ 降级路径 2 — Retriever 超时"]
        direction LR
        T1["retrieve timeout"] --> T2["读 cache"] --> T3["rerank / generate"]
    end

    subgraph FB["🆘 兜底路径 — 无缓存"]
        direction LR
        F1["retriever down"] --> F2["静态兜底消息"]
    end
```

### ResilientOrchestrator 决策流程

```mermaid
flowchart TD
    Start(["handle(query)"]) --> Ret{"Retriever 可用?"}

    Ret -->|"是"| Hits["fake_retrieve<br/>+ 写 cache"]
    Ret -->|"否"| Cache{"cache 有 hits?"}

    Cache -->|"是"| Cached["用缓存 hits<br/>reason: retriever_timeout_used_cache"]
    Cache -->|"否"| Fallback["返回兜底消息<br/>reason: retriever_down_no_cache"]

    Hits --> Rerank{"Reranker 可用?"}
    Cached --> Rerank

    Rerank -->|"是"| Topk["fake_rerank → topk"]
    Rerank -->|"否"| Raw["降级 hits[:3]<br/>reason: reranker_down_using_raw_hits"]

    Topk --> Gen["fake_generate"]
    Raw --> Gen
    Gen --> OK(["返回 answer<br/>degraded + reason"])
    Fallback --> End(["结束"])
```

### 时序：正常 vs 降级

```mermaid
sequenceDiagram
    participant C as Client
    participant O as ResilientOrchestrator
    participant R as Retriever
    participant RR as Reranker
    participant G as Generator

    C->>O: handle(query)

    alt Retriever 正常
        O->>R: fake_retrieve
        R-->>O: hits（写入 cache）
    else Retriever 故障
        O->>O: 读 cache[query]
        Note over O: 无缓存 → 直接返回兜底消息
    end

    alt Reranker 正常
        O->>RR: fake_rerank(hits)
        RR-->>O: topk
    else Reranker 故障
        O->>O: topk = hits[:3]（跳过 rerank）
    end

    O->>G: fake_generate(query, topk)
    G-->>O: answer
    O-->>C: {answer, degraded, reason}
```

In [ ]:
class RerankerUnavailable(Exception): pass
class RetrieverTimeout(Exception): pass

class ResilientOrchestrator:
    def __init__(self, reranker_up: bool = True, retriever_up: bool = True):
        self.reranker_up  = reranker_up
        self.retriever_up = retriever_up
        self._cache: dict[str, list] = {}

    def handle(self, query: str) -> dict:
        degraded_reason = None

        # ── 检索（有缓存则用缓存）────────────────────────────────────────────
        try:
            if not self.retriever_up:
                raise RetrieverTimeout("retriever unavailable")
            hits = fake_retrieve(query, latency=0.1)
            self._cache[query] = hits          # 写缓存
        except RetrieverTimeout:
            hits = self._cache.get(query, [])
            if hits:
                degraded_reason = "retriever_timeout_used_cache"
            else:
                return {
                    "answer": "暂时无法检索，请稍后重试。",
                    "degraded": True,
                    "reason": "retriever_down_no_cache",
                }

        # ── 重排（降级：跳过，直接用粗检结果）───────────────────────────────
        try:
            if not self.reranker_up:
                raise RerankerUnavailable("reranker is down")
            topk = fake_rerank(query, hits, top_k=3, latency=0.1)
        except RerankerUnavailable:
            topk = hits[:3]                    # 降级：直接用 top-3 粗检结果
            degraded_reason = degraded_reason or "reranker_down_using_raw_hits"

        # ── 生成 ─────────────────────────────────────────────────────────────
        answer = fake_generate(query, topk, latency=0.3)
        return {
            "answer": answer[:80],
            "degraded": degraded_reason is not None,
            "reason": degraded_reason,
        }


# ── 测试三种场景 ──────────────────────────────────────────────────────────────
scenarios = [
    ("✅ 正常路径",          dict(reranker_up=True,  retriever_up=True)),
    ("⚠️  Reranker 故障",   dict(reranker_up=False, retriever_up=True)),
    ("🆘 Retriever 故障",   dict(reranker_up=True,  retriever_up=False)),
]

query = "什么是优雅降级？"
for label, flags in scenarios:
    orc = ResilientOrchestrator(**flags)
    r   = orc.handle(query)
    status = "降级" if r["degraded"] else "正常"
    print(f"{label}")
    print(f"  状态：{status}  原因：{r.get('reason', 'none')}")
    print(f"  答案：{r['answer'][:60]}...")
    print()

> **关键观察**：
> - Reranker 故障时，系统仍能返回答案（质量略降）
> - Retriever 故障时，优先用缓存；无缓存才返回兜底消息
> - 每个降级路径都有明确的 `reason`，便于监控告警

### 三种测试场景

```mermaid
flowchart TB
    subgraph S1["✅ 正常路径"]
        S1A["reranker_up=True<br/>retriever_up=True"] --> S1B["retrieve → rerank → generate"]
        S1B --> S1C["degraded=False"]
    end

    subgraph S2["⚠️ Reranker 故障"]
        S2A["reranker_up=False<br/>retriever_up=True"] --> S2B["retrieve → hits[:3] → generate"]
        S2B --> S2C["degraded=True<br/>reason=reranker_down_using_raw_hits"]
    end

    subgraph S3["🆘 Retriever 故障"]
        S3A["reranker_up=True<br/>retriever_up=False"] --> S3B{"有 cache?"}
        S3B -->|"无"| S3C["兜底消息<br/>retriever_down_no_cache"]
        S3B -->|"有"| S3D["cache → rerank → generate"]
    end
```

### 降级层级（质量 ↔ 可用性）

```mermaid
flowchart BT
    L4["L4 正常<br/>retrieve + rerank + generate<br/>质量最高"] 
    L3["L3 Reranker 降级<br/>skip rerank，用粗检 top-3"]
    L2["L2 Retriever 降级<br/>用 cache 中的历史 hits"]
    L1["L1 兜底<br/>静态消息「请稍后重试」"]

    L4 -.->|"Reranker 故障"| L3
    L3 -.->|"Retriever 故障"| L2
    L2 -.->|"无 cache"| L1
```

## 📖 讲义

# Q & A

## 有什么问题？

<br>

**课后作业**：完成练习阶段 1（Redis Pipeline 三 Agent）
提交：代码仓库链接 + 一张端到端 trace 截图

<br>

*下一课：Multi-Agent 评估与持续优化*

---
## 课程小结

| 模式 | 核心机制 | 适用场景 | Demo 实现 |
|------|----------|----------|-----------|
| **Monolithic** | 串行函数调用 | 简单原型 | `monolithic_agent()` |
| **Pipeline** | `queue.Queue` + 线程 Worker | 线性流水、高并发 | `q_retrieve / q_rerank / q_generate` |
| **Hub-and-Spoke** | `asyncio.gather` 并行 | 多角色协作、并行加速 | `orchestrator()` |
| **Blackboard** | 共享字典 + `threading.Event` | 迭代协作、松耦合 | `Blackboard` 类 |
| **幂等** | `request_id` + 已处理集合 | 所有消息驱动场景 | `IdempotentProcessor` |
| **可观测性** | `span()` + `Metrics` | 生产监控、性能诊断 | `traced_pipeline()` |
| **优雅降级** | `try/except` + 降级路径 | 高可用系统 | `ResilientOrchestrator` |

### 三条铁律

1. **每条消息带 `request_id`**，处理前先做幂等检查
2. **每个 Agent 暴露 latency + status**，不监控等于盲飞
3. **提前设计失败路径**，不要等故障发生后再想降级逻辑

---
*下一步：把这些 Agent 换成真实的 LLM API 调用，并接入 Redis / Kafka 消息队列*

---
**导航**：[← 01.6 · 可观测性](./01.6_observability.ipynb) · [📋 课程索引](./01_multi_agent_arch_demo.ipynb)
